# 🩺 Diabetic Retinopathy Grading — Production Pipeline v19
## 30-Step Complete Build | Windows HP Victus (CPU+GPU) | APTOS 2019
---


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 1 — Setup: GPU / CPU / Storage / CUDA / MPS / Memory Check
# ═══════════════════════════════════════════════════════════════════
import os, sys, platform, shutil, subprocess

print("=" * 65)
print("  SYSTEM DIAGNOSTICS — HP Victus Gaming Laptop")
print("=" * 65)
print(f"  OS           : {platform.system()} {platform.release()}")
print(f"  Python       : {sys.version.split()[0]}")
print(f"  Architecture : {platform.machine()}")
print(f"  CPU          : {platform.processor() or 'N/A'}")

# Memory
try:
    import psutil
    ram = psutil.virtual_memory()
    print(f"  RAM Total    : {ram.total / 1e9:.1f} GB")
    print(f"  RAM Available: {ram.available / 1e9:.1f} GB")
except ImportError:
    print("  RAM          : install psutil for details")

# Disk
total, used, free = shutil.disk_usage(os.path.expanduser("~"))
print(f"  Disk Free    : {free / 1e9:.1f} GB {'✅' if free/1e9 > 5 else '⚠️ LOW'}")

# GPU (NVIDIA)
try:
    r = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
         "--format=csv,noheader"],
        capture_output=True, text=True, timeout=10
    )
    if r.returncode == 0:
        for line in r.stdout.strip().split("\n"):
            print(f"  GPU          : {line.strip()}")
    else:
        print("  GPU          : nvidia-smi failed")
except FileNotFoundError:
    print("  GPU          : nvidia-smi not found — CPU-only mode")

# Quick CUDA test
try:
    import torch
    print(f"\n  PyTorch      : {torch.__version__}")
    print(f"  CUDA avail   : {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"  CUDA version : {torch.version.cuda}")
        print(f"  GPU (torch)  : {torch.cuda.get_device_name(0)}")
        vram = torch.cuda.get_device_properties(0).total_mem / 1e9
        print(f"  VRAM         : {vram:.1f} GB")
    mps_avail = hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()
    print(f"  MPS avail    : {mps_avail}")
except ImportError:
    print("  PyTorch      : NOT installed yet (Step 2)")

print("=" * 65)
print("✅ Step 1 complete — system check done.")

## 📦 Step 2 — Install / Upgrade Dependencies


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 2 — Install Requirements
# (timm, albumentations, opencv, sklearn, scipy, tqdm, etc.)
# ═══════════════════════════════════════════════════════════════════
import sys, subprocess

packages = [
    "torch>=2.1", "torchvision", "torchaudio",
    "timm>=1.0.0",
    "albumentations>=1.4.0",
    "opencv-python-headless",
    "scikit-learn",
    "scipy",
    "pandas",
    "numpy",
    "tqdm",
    "matplotlib",
    "grad-cam",
    "streamlit>=1.35.0",
    "huggingface_hub>=0.23.0",
    "kaggle",
    "pyarrow",
    "fastparquet",
    "pillow<11.0",
    "psutil",
    "ipywidgets",
]

print("🚀 Installing / upgrading dependencies...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + packages,
    capture_output=True, text=True
)
if result.returncode == 0:
    print("✅ All dependencies installed successfully!")
else:
    print("⚠️ Some issues during installation:")
    print(result.stderr[-2000:])
print("🎯 Step 2 complete.")

## 📥 Step 3 — Kaggle File Upload & Authentication
Upload your `kaggle.json` file below, or set environment variables.
Get your API token from: https://www.kaggle.com/settings → API → Create New Token


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 3 — Kaggle File Upload & Authentication
# Supports: (1) File upload widget  (2) Manual path  (3) Env vars
# ═══════════════════════════════════════════════════════════════════
import os, sys, json, shutil, platform
from pathlib import Path

KAGGLE_DIR  = Path.home() / ".kaggle"
KAGGLE_JSON = KAGGLE_DIR / "kaggle.json"

def _install_kaggle_json(src_path):
    """Copy kaggle.json to ~/.kaggle/ with correct permissions."""
    KAGGLE_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copy2(str(src_path), str(KAGGLE_JSON))
    if platform.system() != "Windows":
        KAGGLE_JSON.chmod(0o600)
    # Verify it has the right keys
    data = json.loads(KAGGLE_JSON.read_text())
    assert "username" in data and "key" in data, "kaggle.json must have 'username' and 'key'"
    print(f"✅ kaggle.json installed for user: {data['username']}")
    return True

# ── Check if already configured ─────────────────────────────────
if KAGGLE_JSON.exists():
    try:
        d = json.loads(KAGGLE_JSON.read_text())
        print(f"✅ kaggle.json already configured for user: {d.get('username', '?')}")
    except Exception:
        print("⚠️ kaggle.json exists but is invalid. Re-upload below.")
        KAGGLE_JSON.unlink()

if not KAGGLE_JSON.exists():
    # ── Method 1: Environment variables ──────────────────────────
    ku = os.environ.get("KAGGLE_USERNAME", "")
    kk = os.environ.get("KAGGLE_KEY", "")
    if ku and kk:
        KAGGLE_DIR.mkdir(parents=True, exist_ok=True)
        KAGGLE_JSON.write_text(json.dumps({"username": ku, "key": kk}))
        if platform.system() != "Windows":
            KAGGLE_JSON.chmod(0o600)
        print(f"✅ kaggle.json written from env vars (user: {ku})")
    else:
        # ── Method 2: File upload widget (Jupyter) ───────────────
        try:
            import ipywidgets as widgets
            from IPython.display import display, HTML

            display(HTML("<h3>📁 Upload your kaggle.json file:</h3>"))

            upload_widget = widgets.FileUpload(
                accept=".json",
                multiple=False,
                description="kaggle.json"
            )
            status_label = widgets.Label(value="⏳ Waiting for upload...")

            def on_upload_change(change):
                if upload_widget.value:
                    uploaded = list(upload_widget.value.values())[0] if isinstance(upload_widget.value, dict) else upload_widget.value[0]
                    content = uploaded["content"] if isinstance(uploaded, dict) else uploaded.content
                    tmp_path = Path.home() / "_tmp_kaggle.json"
                    tmp_path.write_bytes(content if isinstance(content, bytes) else content.tobytes())
                    try:
                        _install_kaggle_json(tmp_path)
                        status_label.value = "✅ kaggle.json uploaded and installed!"
                    except Exception as e:
                        status_label.value = f"❌ Error: {e}"
                    finally:
                        tmp_path.unlink(missing_ok=True)

            upload_widget.observe(on_upload_change, names="value")
            display(upload_widget)
            display(status_label)

            # ── Method 3: Manual path fallback ───────────────────
            display(HTML("""
            <p style='margin-top:15px;'>
            <b>Alternative:</b> Place your <code>kaggle.json</code> manually at:<br>
            <code>""" + str(KAGGLE_JSON) + """</code><br>
            Then re-run this cell.
            </p>
            """))

        except ImportError:
            print("⚠️ ipywidgets not available for file upload.")
            print(f"   Please place kaggle.json at: {KAGGLE_JSON}")
            print("   Or set KAGGLE_USERNAME and KAGGLE_KEY environment variables.")

# ── Final verification ───────────────────────────────────────────
if KAGGLE_JSON.exists():
    os.environ["KAGGLE_CONFIG_DIR"] = str(KAGGLE_DIR)
    print(f"\n📂 Kaggle config dir: {KAGGLE_DIR}")
    print("✅ Step 3 complete — Kaggle authentication ready.")
else:
    print("\n⚠️ kaggle.json not yet configured.")
    print("   Upload the file above, or place it manually, then re-run this cell.")

## 📥 Step 4 — Dataset Download & Extraction (APTOS 2019)


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 4 — Download & Extract APTOS 2019 Blindness Detection
# ═══════════════════════════════════════════════════════════════════
import os, sys, json, subprocess, zipfile, shutil, platform, warnings
from pathlib import Path

# ── Paths ─────────────────────────────────────────────────────────
DATA_DIR     = Path(os.environ.get("DATA_DIR",     str(Path.home() / "DR_data" / "aptos2019")))
ARTIFACT_DIR = Path(os.environ.get("ARTIFACT_DIR", str(Path.home() / "DR_data" / "artifacts_v19")))
IMG_DIR      = DATA_DIR / "train_images"
CSV_PATH     = DATA_DIR / "train.csv"
PLOT_DIR     = DATA_DIR / "plots_v19"

for d in [DATA_DIR, ARTIFACT_DIR, PLOT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

_dl_flag = ARTIFACT_DIR / "_done_download.flag"

if (_dl_flag.exists()
        and IMG_DIR.exists()
        and CSV_PATH.exists()):
    n = len(list(IMG_DIR.glob("*.png")))
    print(f"✅ [RESUME] Dataset already present — {n:,} images.")
else:
    KAGGLE_JSON = Path.home() / ".kaggle" / "kaggle.json"
    if not KAGGLE_JSON.exists():
        raise FileNotFoundError(
            "Kaggle credentials not configured. Run Step 3 first.\n"
            f"Expected: {KAGGLE_JSON}"
        )

    DATA_DIR.mkdir(parents=True, exist_ok=True)
    ZIP = DATA_DIR / "aptos2019.zip"

    if not ZIP.exists():
        print("📥 Downloading APTOS 2019 dataset (~1.4 GB)...")
        r = subprocess.run(
            [sys.executable, "-m", "kaggle", "competitions", "download",
             "-c", "aptos2019-blindness-detection", "-p", str(DATA_DIR)],
            capture_output=True, text=True
        )
        if r.returncode != 0:
            print("STDERR:", r.stderr[-2000:])
            raise RuntimeError(
                "Kaggle download failed.\n"
                "1. Check credentials  2. Accept competition rules at kaggle.com\n"
                "3. Or manually download & extract to: " + str(DATA_DIR)
            )
        print("✅ Download complete.")
    else:
        print(f"✅ ZIP already present: {ZIP}")

    print("📦 Extracting...")
    with zipfile.ZipFile(ZIP, "r") as zf:
        zf.extractall(DATA_DIR)
    print("✅ Extraction complete.")
    _dl_flag.touch()

n_imgs = len(list(IMG_DIR.glob("*.png"))) if IMG_DIR.exists() else 0
print(f"\n📂 Dataset ready — {n_imgs:,} training images at:")
print(f"   {IMG_DIR}")

## 📊 Step 5 — Load Dataset (train.csv, image paths, label mapping)


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 5 — Load Dataset
# ═══════════════════════════════════════════════════════════════════
import os, sys, io, json, gc, time, random, shutil, warnings, pickle, platform
from pathlib import Path
from copy import deepcopy

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, cohen_kappa_score, ConfusionMatrixDisplay
)

warnings.filterwarnings("ignore")

# ── Reproducibility ───────────────────────────────────────────────
SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
seed_everything()

# ── Device ────────────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = "cuda"
    print(f"🔥 CUDA GPU: {torch.cuda.get_device_name(0)}")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
    print("🍎 Apple MPS detected")
else:
    DEVICE = "cpu"
    print("💻 Running on CPU")

USE_AMP = (DEVICE == "cuda")
print(f"✅ PyTorch {torch.__version__} | timm {timm.__version__}")
print(f"   Device: {DEVICE.upper()}  AMP: {'ON' if USE_AMP else 'OFF'}")

# ── Constants ─────────────────────────────────────────────────────
NUM_CLASSES  = 5
N_FOLDS      = 5
GRADE_MAP    = {0:"No DR", 1:"Mild DR", 2:"Moderate DR", 3:"Severe DR", 4:"Proliferative DR"}
GRADE_COLORS = ["#2ecc71", "#f1c40f", "#e67e22", "#e74c3c", "#8e44ad"]

# ── Safe torch.load ───────────────────────────────────────────────
def safe_load(path, map_location="cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)

# ── Resume State (JSON) ──────────────────────────────────────────
_STATE_FILE = ARTIFACT_DIR / "training_state.json"
def st_load():
    if _STATE_FILE.exists():
        try: return json.loads(_STATE_FILE.read_text())
        except: return {}
    return {}
def st_save(key, val):
    s = st_load(); s[key] = val
    _STATE_FILE.write_text(json.dumps(s, indent=2, default=str))
def st_get(key, default=None):
    return st_load().get(key, default)

# ── Load CSV ──────────────────────────────────────────────────────
if not CSV_PATH.exists():
    raise FileNotFoundError(f"train.csv not found at {CSV_PATH}. Run Step 4 first.")

df = pd.read_csv(CSV_PATH)
df["image_path"]  = df["id_code"].apply(lambda x: str(IMG_DIR / f"{x}.png"))
df["grade_label"] = df["diagnosis"].map(GRADE_MAP)
df["binary"]      = (df["diagnosis"] >= 1).astype(int)

# Verify
missing = df[~df["image_path"].apply(lambda p: Path(p).exists())]
if len(missing):
    print(f"⚠️ {len(missing)} missing images — dropping.")
    df = df[df["image_path"].apply(lambda p: Path(p).exists())].reset_index(drop=True)
else:
    print(f"✅ All {len(df):,} images found.")

print(f"\nClass distribution:")
for g in range(5):
    n = (df["diagnosis"] == g).sum()
    bar = "█" * (n // 50)
    print(f"  Grade {g} ({GRADE_MAP[g]:20s}): {n:5d}  {bar}")
imb = df["diagnosis"].value_counts().max() / df["diagnosis"].value_counts().min()
print(f"\n  Total: {len(df):,}  |  Imbalance ratio: {imb:.1f}x")

## 🧹 Step 6 — Data Cleaning
Removes corrupted, blurry, black, blue/noisy images using Laplacian + intensity checks.


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 6 — Data Cleaning
# ═══════════════════════════════════════════════════════════════════
_clean_flag = ARTIFACT_DIR / "_done_cleaning.flag"
_clean_path = ARTIFACT_DIR / "df_clean.parquet"

if _clean_flag.exists() and _clean_path.exists():
    df = pd.read_parquet(_clean_path)
    df["image_path"]  = df["id_code"].apply(lambda x: str(IMG_DIR / f"{x}.png"))
    df["grade_label"] = df["diagnosis"].map(GRADE_MAP)
    df["binary"]      = (df["diagnosis"] >= 1).astype(int)
    print(f"✅ [RESUME] Cleaned dataset — {len(df):,} rows.")
else:
    print("🧹 Running data cleaning...")

    def _image_quality(path):
        bgr = cv2.imread(str(path))
        if bgr is None:
            return False, "unreadable"
        h, w = bgr.shape[:2]
        if h < 100 or w < 100:
            return False, "too_small"
        gray    = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
        lap_var = cv2.Laplacian(gray, cv2.CV_64F).var()
        bright  = float(gray.mean())
        if bright < 5.0:
            return False, "black_image"
        if lap_var < 30.0:
            return False, "blurry"
        b, g_ch, r = cv2.split(bgr)
        if float(b.mean()) > float(g_ch.mean()) * 1.8:
            return False, "blue_artefact"
        return True, "ok"

    results = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Cleaning"):
        valid, reason = _image_quality(row["image_path"])
        results.append({"id_code": row["id_code"], "valid": valid, "reason": reason})

    res_df  = pd.DataFrame(results)
    invalid = res_df[~res_df["valid"]]
    print(f"  Total: {len(df):,}  |  Removed: {len(invalid):,}")
    for reason, cnt in invalid["reason"].value_counts().items():
        print(f"    {reason}: {cnt}")

    valid_ids = set(res_df[res_df["valid"]]["id_code"])
    df = df[df["id_code"].isin(valid_ids)].reset_index(drop=True)
    df["grade_label"] = df["diagnosis"].map(GRADE_MAP)
    df["binary"]      = (df["diagnosis"] >= 1).astype(int)
    df.to_parquet(_clean_path, index=False)
    _clean_flag.touch()
    print(f"✅ Clean dataset: {len(df):,} images saved.")

print("\nPost-cleaning class distribution:")
for g in range(5):
    n = (df["diagnosis"] == g).sum()
    print(f"  Grade {g} ({GRADE_MAP[g]:20s}): {n:5d}")

## 🔬 Step 7 — Exploratory Data Analysis (EDA)
Class distribution, sample visualization, brightness, size distribution.


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 7 — EDA: Class Distribution + Sample Visualization
# ═══════════════════════════════════════════════════════════════════
%matplotlib inline

_eda_flag = PLOT_DIR / "eda_distribution.png"

if _eda_flag.exists():
    print("✅ [RESUME] EDA plot exists — displaying.")
    img_tmp = plt.imread(str(_eda_flag))
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.imshow(img_tmp); ax.axis("off")
    plt.tight_layout(); plt.show()
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    counts = df["diagnosis"].value_counts().sort_index()
    axes[0].bar([GRADE_MAP[i] for i in range(5)], counts.values,
                color=GRADE_COLORS, edgecolor="k", linewidth=0.6)
    axes[0].set_title("Class Distribution — APTOS 2019", fontweight="bold")
    axes[0].set_ylabel("Count")
    axes[0].tick_params(axis="x", rotation=20)
    for i, v in enumerate(counts.values):
        axes[0].text(i, v + 30, str(v), ha="center", fontsize=10)

    axes[1].pie(counts.values, labels=[GRADE_MAP[i] for i in range(5)],
                colors=GRADE_COLORS, autopct="%1.1f%%", startangle=90)
    axes[1].set_title("Class Proportions", fontweight="bold")

    plt.suptitle(f"APTOS 2019 — {len(df):,} images (post-cleaning)",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(_eda_flag, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"✅ EDA plot saved → {_eda_flag}")

# Sample images per class
_sample_flag = PLOT_DIR / "eda_samples.png"
if not _sample_flag.exists():
    fig2, axes2 = plt.subplots(NUM_CLASSES, 4, figsize=(16, NUM_CLASSES * 3))
    for g in range(NUM_CLASSES):
        subset = df[df["diagnosis"] == g].sample(min(4, (df["diagnosis"] == g).sum()),
                                                  random_state=SEED)
        for j, (_, row) in enumerate(subset.iterrows()):
            img = cv2.imread(row["image_path"])
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            axes2[g][j].imshow(img if img is not None else np.zeros((100,100,3), np.uint8))
            axes2[g][j].axis("off")
            if j == 0:
                axes2[g][j].set_ylabel(f"G{g}: {GRADE_MAP[g]}", fontsize=9,
                                        fontweight="bold", rotation=0, labelpad=80, va="center")
    plt.suptitle("Sample Images per Class", fontweight="bold")
    plt.tight_layout()
    plt.savefig(_sample_flag, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"✅ Sample visualization saved → {_sample_flag}")
else:
    print("✅ [RESUME] Sample visualization exists.")

## 📊 Step 8 — Label Analysis
Class imbalance analysis → required for weighting & sampler.


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 8 — Label / Class Imbalance Analysis
# ═══════════════════════════════════════════════════════════════════
print("Label imbalance analysis:")
vc = df["diagnosis"].value_counts().sort_index()
class_counts = np.bincount(df["diagnosis"].values, minlength=NUM_CLASSES).astype(float)
class_weights_np = len(df) / (NUM_CLASSES * np.maximum(class_counts, 1))
class_weights_np = class_weights_np / class_weights_np.sum() * NUM_CLASSES

for g, c in vc.items():
    pct = c / len(df) * 100
    print(f"  Grade {g}: {c:5d} ({pct:5.1f}%)  weight = {class_weights_np[g]:.3f}")

print(f"\n  Imbalance ratio (max/min): {vc.max() / vc.min():.1f}x")
print("  → WeightedRandomSampler + class-weighted loss are MANDATORY.")
print("  → These weights will be used in Step 16 (loss function).")

## 🖼️ Step 9 — Preprocessing (STRICT PIPELINE)
- Retina crop (remove black borders)
- Resize with padding (aspect ratio preserved)
- Normalize to [0,1]
- Ben Graham enhancement (4×img − 4×blur + 128)
- Optional CLAHE (low probability)


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 9 — Fundus Preprocessing Pipeline
# ═══════════════════════════════════════════════════════════════════
IMG_SIZE = int(os.environ.get("IMG_SIZE", 512))
BG_SIGMA = max((IMG_SIZE // 10) | 1, 1)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def _make_circular_mask(img):
    h, w = img.shape[:2]
    mask = np.zeros((h, w), np.uint8)
    cv2.circle(mask, (w // 2, h // 2), int(min(h, w) // 2 * 0.97), 255, -1)
    return mask

def _clahe_lab(rgb):
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    lab[:, :, 0] = clahe.apply(lab[:, :, 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

def _green_emphasis(rgb):
    out = rgb.copy().astype(np.float32)
    out[:, :, 1] = np.clip(out[:, :, 1] * 1.1, 0, 255)
    return out.astype(np.uint8)

def preprocess_fundus(path_or_array, size=None):
    """
    Full fundus preprocessing:
    1. Load RGB  2. Crop black borders  3. Resize+Pad to square
    4. Circular mask  5. CLAHE (LAB)  6. Ben Graham sharpening
    7. Green emphasis
    Returns: uint8 RGB [size × size × 3]
    """
    target = size or IMG_SIZE
    if isinstance(path_or_array, np.ndarray):
        rgb = path_or_array.copy()
    else:
        bgr = cv2.imread(str(path_or_array))
        if bgr is None:
            return np.zeros((target, target, 3), np.uint8)
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    # Auto-crop black borders
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    _, thresh = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
    coords = cv2.findNonZero(thresh)
    if coords is not None:
        x, y, w, h = cv2.boundingRect(coords)
        rgb = rgb[y:y+h, x:x+w]

    # Resize keeping aspect ratio → pad to square
    h, w = rgb.shape[:2]
    scale = target / max(h, w)
    nh, nw = int(round(h * scale)), int(round(w * scale))
    rgb = cv2.resize(rgb, (nw, nh), interpolation=cv2.INTER_AREA)
    pt = (target - nh) // 2; pb = target - nh - pt
    pl = (target - nw) // 2; pr = target - nw - pl
    rgb = cv2.copyMakeBorder(rgb, pt, pb, pl, pr, cv2.BORDER_REFLECT_101)

    # Circular mask
    mask = _make_circular_mask(rgb)
    rgb[mask == 0] = 0

    # CLAHE in LAB space
    rgb = _clahe_lab(rgb)

    # Ben Graham: vessel sharpening (4×img − 4×blur + 128)
    sig = max((target // 10) | 1, 1)
    blur = cv2.GaussianBlur(rgb, (0, 0), sigmaX=sig)
    rgb = cv2.addWeighted(rgb, 4, blur, -4, 128)
    rgb[mask == 0] = 0

    # Green channel emphasis
    rgb = _green_emphasis(rgb)
    return rgb

print(f"✅ preprocess_fundus defined (IMG_SIZE={IMG_SIZE}, BG_sigma={BG_SIGMA})")

# Benchmark
if len(df) > 0:
    t0 = time.time()
    for _ in range(3):
        preprocess_fundus(df["image_path"].iloc[0])
    lat = (time.time() - t0) / 3 * 1000
    print(f"   Avg latency: {lat:.1f} ms/image {'✅' if lat < 100 else '⚠️ slow'}")

# Visual check
%matplotlib inline
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for i, g in enumerate([0, 1, 2, 4]):
    row = df[df["diagnosis"] == g].iloc[0]
    raw = cv2.cvtColor(cv2.imread(row["image_path"]), cv2.COLOR_BGR2RGB)
    pre = preprocess_fundus(row["image_path"], size=384)
    axes[0][i].imshow(raw); axes[0][i].set_title(f"Raw G{g}"); axes[0][i].axis("off")
    axes[1][i].imshow(pre); axes[1][i].set_title(f"Processed G{g}"); axes[1][i].axis("off")
plt.suptitle("Preprocessing Pipeline — Before vs After", fontweight="bold")
plt.tight_layout(); plt.show()

## 💾 Step 10 — Preprocessing Cache
Save processed images as `.npy` to avoid recomputation.


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 10 — Preprocessing Cache (384px .npy files)
# ═══════════════════════════════════════════════════════════════════
from concurrent.futures import ThreadPoolExecutor, as_completed

CACHE_DIR  = DATA_DIR / "cache_v19"
CACHE_FLAG = ARTIFACT_DIR / "_done_cache.flag"
CACHE_SIZE = 384

if CACHE_FLAG.exists() and CACHE_DIR.exists():
    n_cached = len(list(CACHE_DIR.glob("*.npy")))
    USE_CACHE = (n_cached >= len(df) * 0.95)
    print(f"✅ [RESUME] Cache exists — {n_cached:,} files at {CACHE_SIZE}px. USE_CACHE={USE_CACHE}")
else:
    print(f"Building preprocessing cache at {CACHE_SIZE}px...")
    CACHE_DIR.mkdir(parents=True, exist_ok=True)

    def _cache_one(row):
        dst = CACHE_DIR / f'{row["id_code"]}.npy'
        if dst.exists():
            return True
        try:
            img = preprocess_fundus(row["image_path"], size=CACHE_SIZE)
            np.save(str(dst), img)
            return True
        except Exception:
            return False

    rows = [row for _, row in df.iterrows()]
    ok = fail = 0
    n_workers = min(4, os.cpu_count() or 1)
    with ThreadPoolExecutor(max_workers=n_workers) as ex:
        futs = {ex.submit(_cache_one, r): r["id_code"] for r in rows}
        for f in tqdm(as_completed(futs), total=len(futs), desc="Caching"):
            if f.result(): ok += 1
            else:          fail += 1
    print(f"✅ Cache done — {ok:,} ok, {fail} failed.")
    CACHE_FLAG.touch()
    USE_CACHE = True

print(f"\n   CACHE_DIR : {CACHE_DIR}")
print(f"   USE_CACHE : {USE_CACHE}")

## ✂️ Step 11 — Train / Test Split
Hold-out test set (fold 0), never used during training.


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 11 — Hold-Out Test Split Strategy
# ═══════════════════════════════════════════════════════════════════
# Fold 0 = held-out test set (NEVER seen during training)
# Folds 1-4 = cross-validation training folds
# The actual fold assignment is done in Step 12 via StratifiedKFold.
print("Hold-out test strategy:")
print("  • Fold 0 → HELD-OUT TEST SET (never used during training)")
print("  • Folds 1-4 → used for 4-fold cross-validation training")
print("  • No data leakage between training and evaluation")
print("\n  → K-Fold split created in Step 12.")

## 🔀 Step 12 — K-Fold (StratifiedKFold = 5)
Applied only on training data, preserves class distribution.


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 12 — Stratified K-Fold Split (5 folds)
# ═══════════════════════════════════════════════════════════════════
_splits_path = ARTIFACT_DIR / "kfold_splits.parquet"

if _splits_path.exists():
    df = pd.read_parquet(_splits_path)
    df["image_path"]  = df["id_code"].apply(lambda x: str(IMG_DIR / f"{x}.png"))
    df["grade_label"] = df["diagnosis"].map(GRADE_MAP)
    df["binary"]      = (df["diagnosis"] >= 1).astype(int)
    print(f"✅ [RESUME] K-Fold splits loaded.")
else:
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    df["fold"] = -1
    for fold_idx, (_, val_idx) in enumerate(skf.split(df, df["diagnosis"])):
        df.loc[val_idx, "fold"] = fold_idx
    df.to_parquet(_splits_path, index=False)
    print(f"✅ 5-Fold splits created and saved.")

print(f"\nFold distribution:")
for fold in range(N_FOLDS):
    n  = (df["fold"] == fold).sum()
    gd = df[df["fold"] == fold]["diagnosis"].value_counts().sort_index()
    gs = " | ".join([f"G{g}:{c}" for g, c in gd.items()])
    print(f"  Fold {fold}: {n:5d} samples  [{gs}]")

## 📚 Step 13 — Augmentation + Dataset Pipeline
Albumentations, Custom Dataset classes, WeightedRandomSampler (MANDATORY).


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 13 — Augmentation Pipelines + Dataset Classes + Sampler
# ═══════════════════════════════════════════════════════════════════

# ── Augmentation ─────────────────────────────────────────────────
def build_train_transforms(img_size=IMG_SIZE):
    return A.Compose([
        A.RandomResizedCrop(height=img_size, width=img_size,
                            scale=(0.8, 1.0), ratio=(0.9, 1.1), p=1.0),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.Rotate(limit=15, p=0.7),
        A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.3),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.3),
        A.RandomGamma(gamma_limit=(80, 120), p=0.3),
        A.GaussNoise(var_limit=(10.0, 50.0), p=0.2),
        A.MotionBlur(blur_limit=3, p=0.1),
        A.CoarseDropout(max_holes=8,
                        max_height=img_size // 16, max_width=img_size // 16,
                        min_holes=1, p=0.2),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

def build_val_transforms(img_size=IMG_SIZE):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

def build_tta_transforms(img_size=IMG_SIZE):
    n = A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    r = A.Resize(img_size, img_size)
    return [
        build_val_transforms(img_size),
        A.Compose([A.HorizontalFlip(p=1.0), r, n, ToTensorV2()]),
        A.Compose([A.Rotate(limit=(10, 10), p=1.0), r, n, ToTensorV2()]),
        A.Compose([A.Rotate(limit=(-10, -10), p=1.0), r, n, ToTensorV2()]),
        A.Compose([A.VerticalFlip(p=1.0), r, n, ToTensorV2()]),
    ]

# ── Dataset Classes ──────────────────────────────────────────────
class APTOSDataset(Dataset):
    """5-class grade dataset with cache support."""
    def __init__(self, df, transform=None, img_size=None, use_cache=True):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.img_size = img_size or IMG_SIZE
        self.use_cache = use_cache and USE_CACHE
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]; label = int(row["diagnosis"])
        if self.use_cache:
            cp = CACHE_DIR / f'{row["id_code"]}.npy'
            if cp.exists():
                img = np.load(str(cp))
                if self.img_size != CACHE_SIZE:
                    img = cv2.resize(img, (self.img_size, self.img_size), interpolation=cv2.INTER_AREA)
            else:
                img = preprocess_fundus(row["image_path"], size=self.img_size)
        else:
            img = preprocess_fundus(row["image_path"], size=self.img_size)
        if self.transform:
            img = self.transform(image=img)["image"]
        else:
            img = torch.from_numpy(img.transpose(2, 0, 1)).float() / 255.0
        return img, label

class BinaryAPTOSDataset(Dataset):
    """Stage-1 binary: 0=No DR, 1=any DR."""
    def __init__(self, df, transform=None, img_size=None, use_cache=True):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.img_size = img_size or IMG_SIZE
        self.use_cache = use_cache and USE_CACHE
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        label = torch.tensor(float(row["diagnosis"] >= 1), dtype=torch.float)
        if self.use_cache:
            cp = CACHE_DIR / f'{row["id_code"]}.npy'
            img = np.load(str(cp)) if cp.exists() else preprocess_fundus(row["image_path"], size=self.img_size)
            if cp.exists() and self.img_size != CACHE_SIZE:
                img = cv2.resize(img, (self.img_size, self.img_size), interpolation=cv2.INTER_AREA)
        else:
            img = preprocess_fundus(row["image_path"], size=self.img_size)
        if self.transform:
            img = self.transform(image=img)["image"]
        return img, label

class OrdinalAPTOSDataset(Dataset):
    """Stage-2 ordinal (DR-positive only, grades 1-4)."""
    ORDINAL = {1:[1,0,0,0], 2:[1,1,0,0], 3:[1,1,1,0], 4:[1,1,1,1]}
    def __init__(self, df, transform=None, img_size=None, use_cache=True):
        self.df = df[df["diagnosis"] >= 1].reset_index(drop=True)
        self.transform = transform
        self.img_size = img_size or IMG_SIZE
        self.use_cache = use_cache and USE_CACHE
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]; grade = int(row["diagnosis"])
        ordinal = torch.tensor(self.ORDINAL[grade], dtype=torch.float)
        if self.use_cache:
            cp = CACHE_DIR / f'{row["id_code"]}.npy'
            img = np.load(str(cp)) if cp.exists() else preprocess_fundus(row["image_path"], size=self.img_size)
            if cp.exists() and self.img_size != CACHE_SIZE:
                img = cv2.resize(img, (self.img_size, self.img_size), interpolation=cv2.INTER_AREA)
        else:
            img = preprocess_fundus(row["image_path"], size=self.img_size)
        if self.transform:
            img = self.transform(image=img)["image"]
        return img, ordinal, grade

# ── WeightedRandomSampler (MANDATORY for imbalance) ─────────────
def build_weighted_sampler(df_split):
    labels  = df_split["diagnosis"].values
    counts  = np.bincount(labels, minlength=NUM_CLASSES).astype(float)
    w = 1.0 / np.maximum(counts, 1)
    sample_weights = w[labels]
    return WeightedRandomSampler(
        weights=torch.DoubleTensor(sample_weights),
        num_samples=len(sample_weights), replacement=True
    )

# ── DataLoader Factory ───────────────────────────────────────────
def make_loader(ds, batch_size=16, shuffle=True, num_workers=0,
                drop_last=False, sampler=None):
    if sampler is not None: shuffle = False
    nw = 0 if platform.system() == "Windows" else min(4, os.cpu_count() or 1)
    return DataLoader(
        ds, batch_size=batch_size, shuffle=shuffle, sampler=sampler,
        num_workers=nw, pin_memory=(DEVICE == "cuda"),
        persistent_workers=(nw > 0), drop_last=drop_last
    )

train_transforms   = build_train_transforms(IMG_SIZE)
val_transforms     = build_val_transforms(IMG_SIZE)
tta_transforms_lst = build_tta_transforms(IMG_SIZE)

print(f"✅ Augmentation pipelines (train ops: {len(train_transforms.transforms)}, TTA views: {len(tta_transforms_lst)})")
print(f"✅ Datasets: APTOSDataset, BinaryAPTOSDataset, OrdinalAPTOSDataset")
print(f"✅ WeightedRandomSampler + DataLoader factory ready")
print(f"   num_workers = {'0 (Windows safe)' if platform.system() == 'Windows' else nw}")

## 🔧 Step 14 — DataLoader Creation
Optimized batch size & memory usage for HP Victus GPU.


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 14 — DataLoader Creation + Sanity Check
# ═══════════════════════════════════════════════════════════════════
BS_MAP = {224: 32, 384: 16, 512: 8}

df_train_tmp = df[df["fold"] != 0].reset_index(drop=True)
df_val_tmp   = df[df["fold"] == 0].reset_index(drop=True)

_bs = BS_MAP.get(224, 16)
_tr_ds = APTOSDataset(df_train_tmp, transform=build_val_transforms(224), img_size=224)
_va_ds = APTOSDataset(df_val_tmp,   transform=build_val_transforms(224), img_size=224)
_tr_ld = make_loader(_tr_ds, batch_size=_bs, shuffle=True)
_va_ld = make_loader(_va_ds, batch_size=_bs, shuffle=False)

imgs, labels = next(iter(_tr_ld))
print(f"✅ DataLoader sanity check PASSED")
print(f"   Train: {len(df_train_tmp):,} samples → {len(_tr_ld)} batches")
print(f"   Val:   {len(df_val_tmp):,} samples → {len(_va_ld)} batches")
print(f"   Batch shape: {imgs.shape}  Labels: {labels[:8].tolist()}")
print(f"   Batch sizes: {BS_MAP}")

del _tr_ds, _va_ds, _tr_ld, _va_ld, imgs, labels
gc.collect()

## 🏗️ Step 15 — Model Initialization (MANDATORY BASELINE)
- Backbone: `tf_efficientnetv2_b1` (ImageNet pretrained)
- Head: GlobalAveragePooling → BatchNorm → Dense(256, ReLU) → Dropout(0.5) → 5 classes


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 15 — Model Architecture
# Backbone: tf_efficientnetv2_b1 | Dropout: 0.5 | GeM Pooling
# ═══════════════════════════════════════════════════════════════════
BACKBONE = os.environ.get("BACKBONE", "tf_efficientnetv2_b1")

class GeM(nn.Module):
    """Generalized Mean Pooling."""
    def __init__(self, p=3, eps=1e-6):
        super().__init__()
        self.p   = nn.Parameter(torch.ones(1) * p)
        self.eps = eps
    def forward(self, x):
        return F.avg_pool2d(
            x.clamp(min=self.eps).pow(self.p),
            (x.size(-2), x.size(-1))
        ).pow(1.0 / self.p)

class DRModel(nn.Module):
    """
    DR model: EfficientNetV2-B1 + GeM + custom head.
    mode='softmax' → 5-class | mode='sigmoid' → binary/ordinal
    """
    def __init__(self, backbone=BACKBONE, num_classes=5,
                 dropout=0.5, pretrained=True, mode="softmax"):
        super().__init__()
        self.mode = mode
        self.backbone = timm.create_model(
            backbone, pretrained=pretrained,
            features_only=False, num_classes=0, global_pool=""
        )
        feat_dim = self.backbone.num_features
        self.pool = GeM(p=3)
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.BatchNorm1d(feat_dim),
            nn.Linear(feat_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )
    def forward(self, x):
        return self.head(self.pool(self.backbone(x)))

    def freeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad_(False)

    def unfreeze_top_blocks(self, n=4):
        for p in self.backbone.parameters(): p.requires_grad_(False)
        blocks = list(self.backbone.blocks)
        for block in blocks[-n:]:
            for p in block.parameters(): p.requires_grad_(True)
        for attr in ["conv_head", "bn2", "norm_head"]:
            if hasattr(self.backbone, attr):
                for p in getattr(self.backbone, attr).parameters(): p.requires_grad_(True)

    def unfreeze_all(self):
        for p in self.parameters(): p.requires_grad_(True)

def build_5class_model(pretrained=True):
    return DRModel(BACKBONE, 5, dropout=0.5, pretrained=pretrained, mode="softmax").to(DEVICE)
def build_stage1_model(pretrained=True):
    return DRModel(BACKBONE, 1, dropout=0.5, pretrained=pretrained, mode="sigmoid").to(DEVICE)
def build_stage2_model(pretrained=True):
    return DRModel(BACKBONE, 4, dropout=0.5, pretrained=pretrained, mode="sigmoid").to(DEVICE)

# Sanity forward pass
_m = build_5class_model(pretrained=False)
_x = torch.randn(2, 3, 224, 224).to(DEVICE)
_y = _m(_x)
assert _y.shape == (2, 5), f"Expected (2,5), got {_y.shape}"
print(f"✅ Model forward pass OK: {_x.shape} → {_y.shape}")
print(f"   Backbone    : {BACKBONE}")
print(f"   Dropout     : 0.5")
print(f"   Total params: {sum(p.numel() for p in _m.parameters())/1e6:.2f}M")
del _m, _x, _y; gc.collect()
if DEVICE == "cuda": torch.cuda.empty_cache()

## ⚖️ Step 16 — Loss + Optimizer + Scheduler
- Hybrid: 0.5 × Weighted CE + 0.5 × Focal (γ=2, α=0.25), label smoothing=0.05
- AdamW (weight_decay=1e-4), LR: 3e-4 → 1e-5, Cosine Annealing


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 16 — Loss Functions, MixUp, Metrics
# ═══════════════════════════════════════════════════════════════════

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction="mean"):
        super().__init__()
        self.alpha, self.gamma, self.reduction = alpha, gamma, reduction
    def forward(self, inputs, targets):
        ce = F.cross_entropy(inputs, targets, weight=self.alpha, reduction="none")
        pt = torch.exp(-ce)
        loss = ((1 - pt) ** self.gamma) * ce
        return loss.mean() if self.reduction == "mean" else loss.sum()

class BinaryFocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha, self.gamma = alpha, gamma
    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        prob = torch.sigmoid(logits)
        p_t = prob * targets + (1 - prob) * (1 - targets)
        a_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        return (a_t * (1 - p_t) ** self.gamma * bce).mean()

def compute_class_weights(labels, num_classes=5):
    counts = np.bincount(labels, minlength=num_classes).astype(float)
    weights = len(labels) / (num_classes * np.maximum(counts, 1))
    weights = weights / weights.sum() * num_classes
    return torch.tensor(weights, dtype=torch.float)

def mixup_data(x, y, alpha=0.4, device=DEVICE):
    lam = max(np.random.beta(alpha, alpha), 1 - np.random.beta(alpha, alpha)) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0)).to(device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

def qwk(y_true, y_pred):
    return cohen_kappa_score(y_true, y_pred, weights="quadratic")

def accuracy(y_true, y_pred):
    return (np.array(y_true) == np.array(y_pred)).mean()

def build_5class_criterion(df_split, device=DEVICE):
    """Hybrid 0.5×WCE + 0.5×Focal, label_smoothing=0.05."""
    cw = compute_class_weights(df_split["diagnosis"].values).to(device)
    ce = nn.CrossEntropyLoss(weight=cw, label_smoothing=0.05)
    fl = FocalLoss(alpha=cw, gamma=2.0)
    def criterion(logits, labels):
        return 0.5 * ce(logits, labels) + 0.5 * fl(logits, labels)
    return criterion

print("✅ Loss: Hybrid 0.5×WCE + 0.5×Focal (γ=2, label_smoothing=0.05)")
print("   Optimizer: AdamW (weight_decay=1e-4)")
print("   Scheduler: CosineAnnealingLR")
print("   Metrics: qwk(), accuracy()")

## 💾 Step 17 — Checkpoint & Resume System (FULL RECOVERY)


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 17 — Checkpoint System: epoch, batch, phase, model, optimizer,
# scheduler, global step, best QWK, training history
# ═══════════════════════════════════════════════════════════════════
print("✅ Checkpoint system stores:")
print("   • epoch, batch index, phase")
print("   • model state_dict")
print("   • optimizer state_dict")
print("   • scheduler state_dict")
print("   • global step, best QWK")
print("   • full training history")
print("   → Implemented inline in Step 20 training loop.")
print("   → Phase-level + fold-level checkpoints saved to ARTIFACT_DIR.")
print(f"   → {ARTIFACT_DIR}")

## 📋 Step 18 — Training State Management


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 18 — Training State Management
# ═══════════════════════════════════════════════════════════════════
st_save("pipeline_version", "v19")
assert st_get("pipeline_version") == "v19"

print("✅ Training state management:")
print("   • JSON-based (survives kernel restarts)")
print("   • Tracks epoch-wise loss & QWK")
print("   • Tracks fold-wise performance")
print("   • Saves best model (based on QWK)")
print("   • Enables exact resume (epoch + batch + phase)")
print(f"   • State file: {_STATE_FILE}")

## 🔁 Step 19 — Cross-Validation Training (5-Fold Execution)


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 19 — Cross-Validation Training Configuration
# ═══════════════════════════════════════════════════════════════════
LR           = 3e-4
LR_FINE      = 1e-4
LR_FULL      = 3e-5
WEIGHT_DECAY = 1e-4
USE_MIXUP    = True
GRAD_ACCUM   = 1

PHASES = [
    {"size": 224, "epochs": 15, "name": "P1-Freeze",  "unfreeze": 0},
    {"size": 384, "epochs": 40, "name": "P2-Partial", "unfreeze": 4},
    {"size": 512, "epochs": 25, "name": "P3-Full",    "unfreeze": 99},
]
BS_MAP       = {224: 32, 384: 16, 512: 8}
ES_PATIENCE  = 5
ES_MIN_DELTA = 0.001

print("Cross-Validation Configuration:")
print(f"  Folds: {N_FOLDS} | Backbone: {BACKBONE}")
print(f"  Phases: {[(p['name'], p['size'], p['epochs']) for p in PHASES]}")
print(f"  LR: {LR} → {LR_FINE} → {LR_FULL}")
print(f"  MixUp: {USE_MIXUP} | Grad Accum: {GRAD_ACCUM}")
print(f"  Early Stopping: patience={ES_PATIENCE}, min_delta={ES_MIN_DELTA}")
print(f"  Batch sizes: {BS_MAP}")
print("✅ Ready for training.")

## 🏋️ Step 20 — Training (PHASE-WISE STRATEGY)
- Phase 1: 224px, 15ep, backbone frozen
- Phase 2: 384px, 40ep, partial unfreeze
- Phase 3: 512px, 25ep, full unfreeze


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEPS 19-22 — 5-Fold × 3-Phase Training + Validation + Early Stopping
# + OOF Predictions + Checkpoint Resume
# ═══════════════════════════════════════════════════════════════════
LR           = 3e-4
LR_FINE      = 1e-4
LR_FULL      = 3e-5
WEIGHT_DECAY = 1e-4
USE_MIXUP    = True
GRAD_ACCUM   = 1

PHASES = [
    {"size": 224, "epochs": 15, "name": "P1-Freeze",  "unfreeze": 0},
    {"size": 384, "epochs": 40, "name": "P2-Partial", "unfreeze": 4},
    {"size": 512, "epochs": 25, "name": "P3-Full",    "unfreeze": 99},
]
BS_MAP       = {224: 32, 384: 16, 512: 8}
ES_PATIENCE  = 5
ES_MIN_DELTA = 0.001

oof_probs  = np.zeros((len(df), NUM_CLASSES), dtype=np.float32)
oof_labels = df["diagnosis"].values.copy()
fold_val_qwks = []

print("=" * 70)
print("  5-FOLD × 3-PHASE TRAINING (Steps 19-22)")
print(f"  Backbone : {BACKBONE}  |  Device : {DEVICE.upper()}")
print(f"  Phases   : {[(p['name'], p['size'], p['epochs']) for p in PHASES]}")
print(f"  ES       : patience={ES_PATIENCE}, min_delta={ES_MIN_DELTA}")
print("=" * 70)

for fold in range(N_FOLDS):
    fold_ckpt = ARTIFACT_DIR / f"fold{fold}_best.pt"
    fold_oof  = ARTIFACT_DIR / f"fold{fold}_oof.npy"
    fold_flag = ARTIFACT_DIR / f"_done_fold{fold}.flag"

    # ── Resume ────────────────────────────────────────────────────
    if fold_flag.exists() and fold_ckpt.exists():
        if fold_oof.exists():
            val_idx = df[df["fold"] == fold].index
            oof_probs[val_idx] = np.load(str(fold_oof))
        prev = safe_load(fold_ckpt, "cpu")
        fold_val_qwks.append(prev.get("val_qwk", 0.0))
        print(f"  ✅ [RESUME] Fold {fold} — QWK={fold_val_qwks[-1]:.4f}")
        continue

    print(f"\n  ══ FOLD {fold} ══")
    seed_everything(SEED + fold)

    df_tr = df[df["fold"] != fold].reset_index(drop=True)
    df_va = df[df["fold"] == fold].reset_index(drop=True)
    val_idx = df[df["fold"] == fold].index

    criterion_fn = build_5class_criterion(df_tr)
    model = build_5class_model(pretrained=True)

    best_val_qwk = -1.0
    best_state = None
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_qwk": []}

    for phase_idx, phase in enumerate(PHASES):
        sz, n_ep = phase["size"], phase["epochs"]
        pname, unf = phase["name"], phase["unfreeze"]
        bs = BS_MAP[sz]
        phase_ckpt = ARTIFACT_DIR / f"fold{fold}_phase{phase_idx}.pt"

        if unf == 0:      model.freeze_backbone()
        elif unf >= 99:   model.unfreeze_all()
        else:             model.unfreeze_top_blocks(unf)

        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"  [{pname}] {sz}px × {n_ep}ep | trainable: {trainable/1e6:.2f}M")

        tr_ds = APTOSDataset(df_tr, transform=build_train_transforms(sz), img_size=sz)
        va_ds = APTOSDataset(df_va, transform=build_val_transforms(sz), img_size=sz)
        sampler = build_weighted_sampler(df_tr)
        tr_ld = make_loader(tr_ds, batch_size=bs, sampler=sampler, drop_last=True)
        va_ld = make_loader(va_ds, batch_size=bs, shuffle=False)

        base_lr = {0: LR, 1: LR_FINE, 2: LR_FULL}[phase_idx]
        optimizer = torch.optim.AdamW([
            {"params": model.head.parameters(),  "lr": base_lr},
            {"params": model.pool.parameters(),  "lr": base_lr},
            {"params": [p for p in model.backbone.parameters() if p.requires_grad],
             "lr": base_lr / 10},
        ], weight_decay=WEIGHT_DECAY)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=n_ep, eta_min=base_lr / 100)

        # Resume from phase checkpoint
        phase_start = 0
        if phase_ckpt.exists():
            ps = safe_load(phase_ckpt, DEVICE)
            model.load_state_dict(ps["model_state"])
            optimizer.load_state_dict(ps["optimizer_state"])
            scheduler.load_state_dict(ps["scheduler_state"])
            phase_start  = ps["epoch"]
            best_val_qwk = ps.get("best_val_qwk", best_val_qwk)
            history      = ps.get("history", history)
            if ps.get("best_model_state"):
                best_state = ps["best_model_state"]
            print(f"    Resuming from epoch {phase_start + 1}/{n_ep}")

        model.to(DEVICE)
        scaler = torch.amp.GradScaler("cuda") if USE_AMP else None
        es_counter = 0

        for ep in range(phase_start, n_ep):
            # ── Train ─────────────────────────────────────────────
            model.train()
            ep_loss, ep_preds, ep_labs = 0.0, [], []
            optimizer.zero_grad()

            for step, (imgs, labels) in enumerate(
                    tqdm(tr_ld, desc=f"    {pname} Ep{ep+1:02d}", leave=False)):
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                if USE_MIXUP:
                    imgs, ya, yb, lam = mixup_data(imgs, labels, device=DEVICE)

                if scaler is not None:
                    with torch.amp.autocast("cuda"):
                        logits = model(imgs)
                        loss = (mixup_criterion(criterion_fn, logits, ya, yb, lam)
                                if USE_MIXUP else criterion_fn(logits, labels))
                        loss = loss / GRAD_ACCUM
                    scaler.scale(loss).backward()
                    if (step + 1) % GRAD_ACCUM == 0:
                        scaler.unscale_(optimizer)
                        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                        scaler.step(optimizer); scaler.update(); optimizer.zero_grad()
                else:
                    logits = model(imgs)
                    loss = (mixup_criterion(criterion_fn, logits, ya, yb, lam)
                            if USE_MIXUP else criterion_fn(logits, labels))
                    (loss / GRAD_ACCUM).backward()
                    if (step + 1) % GRAD_ACCUM == 0:
                        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                        optimizer.step(); optimizer.zero_grad()

                ep_loss += loss.item() * GRAD_ACCUM
                ep_preds.extend(logits.argmax(1).detach().cpu().tolist())
                ep_labs.extend(labels.cpu().tolist())

            scheduler.step()
            tr_loss = ep_loss / len(tr_ld)
            tr_acc = accuracy(ep_labs, ep_preds)

            # ── Validate ──────────────────────────────────────────
            model.eval()
            v_loss, v_preds, v_labs = 0.0, [], []
            with torch.no_grad():
                for imgs, labels in va_ld:
                    imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                    logits = model(imgs)
                    v_loss += criterion_fn(logits, labels).item()
                    v_preds.extend(logits.argmax(1).cpu().tolist())
                    v_labs.extend(labels.cpu().tolist())
            v_loss /= max(len(va_ld), 1)
            val_kappa = qwk(v_labs, v_preds)

            history["train_loss"].append(tr_loss)
            history["train_acc"].append(tr_acc)
            history["val_loss"].append(v_loss)
            history["val_qwk"].append(val_kappa)

            flag_str = ""
            if val_kappa > best_val_qwk + ES_MIN_DELTA:
                best_val_qwk = val_kappa
                best_state = deepcopy(model.state_dict())
                es_counter = 0
                flag_str = " ✅ BEST"
            else:
                es_counter += 1

            print(f"    {pname} Ep{ep+1:02d}: "
                  f"TrL={tr_loss:.4f} TrA={tr_acc:.3f} | "
                  f"VaL={v_loss:.4f} QWK={val_kappa:.4f}{flag_str} "
                  f"[ES {es_counter}/{ES_PATIENCE}]")

            # Save phase checkpoint
            torch.save({
                "epoch": ep + 1, "model_state": model.state_dict(),
                "best_model_state": best_state,
                "optimizer_state": optimizer.state_dict(),
                "scheduler_state": scheduler.state_dict(),
                "best_val_qwk": best_val_qwk, "history": history,
                "fold": fold, "phase": phase_idx,
            }, phase_ckpt)

            if es_counter >= ES_PATIENCE:
                print(f"    ⏹ Early stopping at epoch {ep + 1}")
                break

    # ── OOF predictions with TTA ──────────────────────────────────
    if best_state is None: best_state = model.state_dict()
    model.load_state_dict(best_state); model.eval()
    oof_fold = np.zeros((len(df_va), NUM_CLASSES), dtype=np.float32)
    with torch.no_grad():
        for tta_tf in tta_transforms_lst:
            va_tta = APTOSDataset(df_va, transform=tta_tf, img_size=384)
            ld2 = make_loader(va_tta, batch_size=BS_MAP[384], shuffle=False)
            bp = [F.softmax(model(imgs.to(DEVICE)), 1).cpu().numpy() for imgs, _ in ld2]
            oof_fold += np.concatenate(bp)
    oof_fold /= len(tta_transforms_lst)
    oof_probs[val_idx] = oof_fold
    np.save(str(fold_oof), oof_fold)
    fold_val_qwks.append(best_val_qwk)

    torch.save({
        "model_state": best_state, "val_qwk": best_val_qwk,
        "backbone": BACKBONE, "img_size": 384, "fold": fold,
        "history": history, "num_classes": NUM_CLASSES,
    }, fold_ckpt)
    fold_flag.touch()
    print(f"  ✅ Fold {fold} done — Best QWK={best_val_qwk:.4f}")
    del model; gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()

# ── OOF summary ──────────────────────────────────────────────────
oof_preds = oof_probs.argmax(1)
oof_qwk_v = qwk(oof_labels, oof_preds)
oof_acc_v = accuracy(oof_labels, oof_preds)
print("\n" + "=" * 70)
print("  K-FOLD RESULTS")
for i, q in enumerate(fold_val_qwks):
    print(f"  Fold {i}: QWK={q:.4f}")
print(f"  Mean   : {np.mean(fold_val_qwks):.4f} ± {np.std(fold_val_qwks):.4f}")
print(f"  OOF QWK: {oof_qwk_v:.4f}  |  OOF Acc: {oof_acc_v*100:.2f}%")
print("=" * 70)
np.save(str(ARTIFACT_DIR / "oof_probs.npy"), oof_probs)
np.save(str(ARTIFACT_DIR / "oof_labels.npy"), oof_labels)
st_save("oof_qwk", float(oof_qwk_v))

## 📈 Step 21 — Validation (QWK per epoch per fold)


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 21 — Validation Results Summary (QWK per fold)
# ═══════════════════════════════════════════════════════════════════
print("Validation QWK per fold:")
for i, q in enumerate(fold_val_qwks):
    marker = " ← best" if i == int(np.argmax(fold_val_qwks)) else ""
    print(f"  Fold {i}: QWK = {q:.4f}{marker}")
print(f"\n  Mean: {np.mean(fold_val_qwks):.4f} ± {np.std(fold_val_qwks):.4f}")
print(f"  OOF QWK (argmax): {st_get('oof_qwk', 'N/A')}")

## ⏹ Step 22 — Early Stopping (patience=5, min_delta=0.001)


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 22 — Early Stopping (MANDATORY)
# ═══════════════════════════════════════════════════════════════════
print("Early Stopping Configuration:")
print(f"  Monitor:    Validation QWK")
print(f"  Patience:   {ES_PATIENCE} epochs")
print(f"  Min Delta:  {ES_MIN_DELTA}")
print(f"  Action:     Restore best weights → next phase or end fold")
print("\n  ✅ Integrated into Step 20 training loop.")
print("  Early stopping triggered per-phase within each fold.")

## 🎯 Step 23 — OOF Predictions
Store fold-wise predictions for threshold optimization.


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 23 — OOF Predictions Inspection
# ═══════════════════════════════════════════════════════════════════
oof_probs_loaded = np.load(str(ARTIFACT_DIR / "oof_probs.npy"))
oof_labels_loaded = np.load(str(ARTIFACT_DIR / "oof_labels.npy"))

print(f"OOF Predictions:")
print(f"  Shape:  {oof_probs_loaded.shape}")
print(f"  Argmax QWK:      {qwk(oof_labels_loaded, oof_probs_loaded.argmax(1)):.4f}")
print(f"  Argmax Accuracy:  {accuracy(oof_labels_loaded, oof_probs_loaded.argmax(1))*100:.2f}%")

print("\n  Mean predicted prob per true class:")
for g in range(NUM_CLASSES):
    mask = oof_labels_loaded == g
    if mask.sum() > 0:
        mp = oof_probs_loaded[mask].mean(axis=0)
        print(f"    G{g}: {np.round(mp, 3)}")

## 🔄 Step 24 — TTA (Test-Time Augmentation)
Horizontal flip, mild brightness variation, average predictions.


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 24 — TTA Configuration
# ═══════════════════════════════════════════════════════════════════
print("TTA (Test-Time Augmentation) Configuration:")
print(f"  Views: {len(tta_transforms_lst)}")
print("    1. Original (resize + normalize)")
print("    2. Horizontal flip")
print("    3. Rotate +10°")
print("    4. Rotate -10°")
print("    5. Vertical flip")
print("  Strategy: Average softmax probabilities across views.")
print("  ✅ Already built in Step 13 and used in Steps 19-22.")

## 🎯 Step 25 — Threshold Optimization
Optimize thresholds (NOT argmax) using OOF predictions to maximize QWK.


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 25 — Threshold Optimisation on OOF Predictions
# ═══════════════════════════════════════════════════════════════════
from scipy.optimize import minimize

def optimise_thresholds(probs, labels, n_classes=5):
    base_preds = probs.argmax(1)
    best_qwk = qwk(labels, base_preds)
    best_preds = base_preds.copy()
    print(f"  Argmax QWK (baseline): {best_qwk:.4f}")

    # Sweep class-0 bias
    for bias in np.linspace(-0.4, 0.4, 17):
        biased = probs.copy(); biased[:, 0] += bias
        p = biased.argmax(1); q = qwk(labels, p)
        if q > best_qwk:
            best_qwk = q; best_preds = p.copy()
            print(f"  Improved QWK={q:.4f} with class-0 bias={bias:.3f}")

    # Nelder-Mead
    def neg_qwk_biased(biases):
        b = probs.copy()
        for c, bv in enumerate(biases): b[:, c] += bv
        return -qwk(labels, b.argmax(1))

    res = minimize(neg_qwk_biased, x0=np.zeros(n_classes),
                   method="Nelder-Mead", options={"maxiter": 500, "xatol": 1e-4, "fatol": 1e-4})
    opt_biased = probs.copy()
    for c, bv in enumerate(res.x): opt_biased[:, c] += bv
    opt_preds = opt_biased.argmax(1)
    opt_q = qwk(labels, opt_preds)
    if opt_q > best_qwk:
        best_qwk = opt_q; best_preds = opt_preds.copy()
        print(f"  Nelder-Mead improved QWK={opt_q:.4f}  biases={np.round(res.x, 3)}")

    return best_preds, best_qwk

print("Threshold optimisation on OOF predictions:")
opt_preds, opt_qwk = optimise_thresholds(oof_probs_loaded, oof_labels_loaded)
print(f"\n  ✅ Optimised OOF QWK: {opt_qwk:.4f}")
print(f"     OOF Accuracy:     {accuracy(oof_labels_loaded, opt_preds)*100:.2f}%")

np.save(str(ARTIFACT_DIR / "oof_opt_preds.npy"), opt_preds)
st_save("opt_qwk", float(opt_qwk))

## 🧪 Step 26 — Testing (Final Hold-Out Set)
Ensemble + TTA + optimized thresholds. No data leakage.


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 26 — Ensemble + TTA on Hold-Out Test (Fold 0)
# ═══════════════════════════════════════════════════════════════════
TEST_FOLD = 0
df_test = df[df["fold"] == TEST_FOLD].reset_index(drop=True)
print(f"Ensemble inference on fold {TEST_FOLD} ({len(df_test):,} samples)...")

ensemble_probs = np.zeros((len(df_test), NUM_CLASSES), dtype=np.float32)
n_models = 0

for fold in range(1, N_FOLDS):
    fold_ckpt = ARTIFACT_DIR / f"fold{fold}_best.pt"
    if not fold_ckpt.exists():
        print(f"  ⚠️ fold{fold}_best.pt not found — skipping")
        continue
    ckpt = safe_load(fold_ckpt, DEVICE)
    model = build_5class_model(pretrained=False)
    model.load_state_dict(ckpt["model_state"]); model.eval()
    img_sz = ckpt.get("img_size", 384)

    fold_probs = np.zeros((len(df_test), NUM_CLASSES), dtype=np.float32)
    with torch.no_grad():
        for tta_tf in build_tta_transforms(img_sz):
            ds = APTOSDataset(df_test, transform=tta_tf, img_size=img_sz)
            ld = make_loader(ds, batch_size=BS_MAP.get(img_sz, 16), shuffle=False)
            bps = []
            for imgs, _ in tqdm(ld, desc=f"  fold{fold} TTA", leave=False):
                bps.append(F.softmax(model(imgs.to(DEVICE)), 1).cpu().numpy())
            fold_probs += np.concatenate(bps, axis=0)
    fold_probs /= len(tta_transforms_lst)
    ensemble_probs += fold_probs; n_models += 1
    print(f"  Fold {fold}: QWK={qwk(df_test['diagnosis'].values, fold_probs.argmax(1)):.4f}")
    del model; gc.collect()

if n_models > 0:
    ensemble_probs /= n_models
    test_preds = ensemble_probs.argmax(1)
    test_labels = df_test["diagnosis"].values
    test_qwk_v = qwk(test_labels, test_preds)
    test_acc_v = accuracy(test_labels, test_preds)
    print(f"\n  ✅ Ensemble ({n_models} models × {len(tta_transforms_lst)} TTA)")
    print(f"     Test QWK:      {test_qwk_v:.4f}")
    print(f"     Test Accuracy: {test_acc_v*100:.2f}%")
    np.save(str(ARTIFACT_DIR / "ensemble_probs.npy"), ensemble_probs)
    np.save(str(ARTIFACT_DIR / "test_preds.npy"), test_preds)
    np.save(str(ARTIFACT_DIR / "test_labels.npy"), test_labels)
    st_save("test_qwk", float(test_qwk_v))
    st_save("test_acc", float(test_acc_v))
else:
    print("⚠️ No checkpoints found. Run training first.")

## 📊 Step 27 — Metrics & Evaluation
QWK (target ≥ 0.90), Accuracy (85-90%), Confusion matrix, Per-class recall.


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 27 — Metrics & Evaluation: Confusion Matrix + Per-Class Recall
# ═══════════════════════════════════════════════════════════════════
%matplotlib inline

_cm_path = PLOT_DIR / "oof_confusion_matrix.png"
oof_probs_l = np.load(str(ARTIFACT_DIR / "oof_probs.npy"))
oof_labels_l = np.load(str(ARTIFACT_DIR / "oof_labels.npy"))
oof_pred_l = oof_probs_l.argmax(1)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

cm = confusion_matrix(oof_labels_l, oof_pred_l)
disp = ConfusionMatrixDisplay(cm, display_labels=[f"G{i}" for i in range(5)])
disp.plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title(
    f"OOF Confusion Matrix\nQWK={qwk(oof_labels_l, oof_pred_l):.4f}  "
    f"Acc={accuracy(oof_labels_l, oof_pred_l)*100:.1f}%", fontweight="bold")

per_class_recall = cm.diagonal() / np.maximum(cm.sum(axis=1), 1)
axes[1].bar([GRADE_MAP[i] for i in range(5)], per_class_recall,
            color=GRADE_COLORS, edgecolor="k")
axes[1].axhline(y=0.85, color="red", linestyle="--", label="Target 85%")
axes[1].set_ylim(0, 1.05); axes[1].set_ylabel("Recall")
axes[1].set_title("Per-Class Recall (OOF)", fontweight="bold")
axes[1].tick_params(axis="x", rotation=20); axes[1].legend()
for i, v in enumerate(per_class_recall):
    axes[1].text(i, v + 0.01, f"{v:.2f}", ha="center", fontsize=9)

plt.tight_layout()
plt.savefig(_cm_path, dpi=150, bbox_inches="tight")
plt.show()

print(f"\nClassification Report (OOF):")
print(classification_report(oof_labels_l, oof_pred_l,
      target_names=[GRADE_MAP[i] for i in range(5)]))

## 📦 Step 28 — Model Export
Save final model + optimized thresholds + label mapping.


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 28 — Model Export
# ═══════════════════════════════════════════════════════════════════
import shutil as _sh

EXPORT_DIR  = ARTIFACT_DIR / "export_v19"
EXPORT_FLAG = ARTIFACT_DIR / "_done_export.flag"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

if EXPORT_FLAG.exists():
    print("✅ [RESUME] Model export already done.")
else:
    _bf = int(np.argmax(fold_val_qwks)) if fold_val_qwks else 0
    _src = ARTIFACT_DIR / f"fold{_bf}_best.pt"
    if _src.exists():
        _sh.copy2(_src, EXPORT_DIR / "best_model.pt")
        print(f"best_model.pt saved (fold {_bf})")

    thr_data = {"stage1_threshold": st_get("stage1_threshold", 0.5),
                "oof_qwk": st_get("oof_qwk"), "opt_qwk": st_get("opt_qwk")}
    label_map = {"grade_map": {str(k): v for k, v in GRADE_MAP.items()},
                 "num_classes": NUM_CLASSES, "backbone": BACKBONE, "img_size": 384}
    meta = {"fold_qwks": fold_val_qwks, "oof_qwk": st_get("oof_qwk"),
            "opt_qwk": st_get("opt_qwk"), "test_qwk": st_get("test_qwk"),
            "device": DEVICE}

    (EXPORT_DIR / "thresholds.json").write_text(json.dumps(thr_data, indent=2))
    (EXPORT_DIR / "label_map.json").write_text(json.dumps(label_map, indent=2))
    (EXPORT_DIR / "metadata.json").write_text(json.dumps(meta, indent=2))

    EXPORT_FLAG.touch()
    print(f"\n✅ Export → {EXPORT_DIR}")
    for f in sorted(EXPORT_DIR.rglob("*")):
        if f.is_file():
            print(f"  {f.name}  ({f.stat().st_size / 1e3:.1f} KB)")

## 🎨 Step 29 — Grad-CAM++ Explainability
Visualize model attention on lesions. Validate clinical relevance.


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 29 — Grad-CAM++ Explainability
# ═══════════════════════════════════════════════════════════════════
%matplotlib inline

try:
    from pytorch_grad_cam import GradCAMPlusPlus
    from pytorch_grad_cam.utils.image import show_cam_on_image
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    _GRADCAM_OK = True
except ImportError:
    print("⚠️ grad-cam not installed. Run: pip install grad-cam")
    _GRADCAM_OK = False

if _GRADCAM_OK and fold_val_qwks:
    _gcam_out = PLOT_DIR / "gradcam_board_v19.png"

    best_fold = int(np.argmax(fold_val_qwks))
    ckpt = safe_load(ARTIFACT_DIR / f"fold{best_fold}_best.pt", "cpu")
    gcam_model = build_5class_model(pretrained=False)
    gcam_model.load_state_dict(ckpt["model_state"])
    gcam_model.eval(); gcam_model.to("cpu")

    target_layers = [gcam_model.backbone.blocks[-1][-1]]
    cam = GradCAMPlusPlus(model=gcam_model, target_layers=target_layers)

    n_per_class = 2
    fig, axes = plt.subplots(NUM_CLASSES, n_per_class * 2, figsize=(14, NUM_CLASSES * 3))

    for grade in range(NUM_CLASSES):
        samples = df[df["diagnosis"] == grade].sample(
            min(n_per_class, (df["diagnosis"] == grade).sum()), random_state=SEED)
        for j, (_, row) in enumerate(samples.iterrows()):
            img_np = preprocess_fundus(row["image_path"], size=384)
            tf = build_val_transforms(384)
            inp = tf(image=img_np)["image"].unsqueeze(0)
            rgb_float = img_np.astype(np.float32) / 255.0
            with torch.no_grad():
                pred_class = gcam_model(inp).argmax(1).item()
            grayscale = cam(input_tensor=inp, targets=[ClassifierOutputTarget(grade)])
            cam_img = show_cam_on_image(rgb_float, grayscale[0], use_rgb=True)

            axes[grade][j*2].imshow(img_np); axes[grade][j*2].axis("off")
            if j == 0:
                axes[grade][j*2].set_ylabel(f"G{grade}\n{GRADE_MAP[grade]}",
                    fontsize=9, fontweight="bold", color=GRADE_COLORS[grade],
                    rotation=0, labelpad=60, va="center")
            axes[grade][j*2+1].imshow(cam_img); axes[grade][j*2+1].axis("off")
            axes[grade][j*2+1].set_title(f"Pred: G{pred_class}", fontsize=8,
                color="green" if pred_class == grade else "red")

    plt.suptitle(f"Grad-CAM++ — Best Fold {best_fold} (QWK={fold_val_qwks[best_fold]:.4f})",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(_gcam_out, dpi=130, bbox_inches="tight")
    plt.show()
    print(f"✅ Grad-CAM++ board saved → {_gcam_out}")
    del gcam_model; gc.collect()
elif _GRADCAM_OK:
    print("⚠️ fold_val_qwks not available — run training first.")

## 🚀 Step 30 — Deployment
Streamlit app + Hugging Face deployment + stable inference pipeline.


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 30 — Deployment: Streamlit App + HF Spaces + Final Summary
# ═══════════════════════════════════════════════════════════════════
import shutil as _sh

DEPLOY_DIR = Path.home() / "DR_data" / "deploy_v19"
DEPLOY_DIR.mkdir(parents=True, exist_ok=True)

# Copy artefacts
_exp_src = ARTIFACT_DIR / "export_v19"
if _exp_src.exists():
    for _f in list(_exp_src.glob("*.pt")) + list(_exp_src.glob("*.json")):
        _sh.copy2(_f, DEPLOY_DIR / _f.name)
    print("✅ Artefacts copied from export_v19/")
else:
    _bf = int(np.argmax(fold_val_qwks)) if fold_val_qwks else 0
    _src = ARTIFACT_DIR / f"fold{_bf}_best.pt"
    if _src.exists():
        _sh.copy2(_src, DEPLOY_DIR / "best_model.pt")
        print(f"best_model.pt copied (fold {_bf})")

# ── model_utils.py ────────────────────────────────────────────────
model_utils_code = """
import os, json
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
GRADE_MAP     = {0:"No DR",1:"Mild DR",2:"Moderate DR",3:"Severe DR",4:"Proliferative DR"}
GRADE_COLORS  = ["#2ecc71","#f1c40f","#e67e22","#e74c3c","#8e44ad"]
NUM_CLASSES   = 5
IMG_SIZE      = 384
BACKBONE      = "tf_efficientnetv2_b1"

def get_device():
    if torch.cuda.is_available(): return "cuda"
    if hasattr(torch.backends,"mps") and torch.backends.mps.is_available(): return "mps"
    return "cpu"
DEVICE = get_device()

def preprocess_fundus(img_array, size=IMG_SIZE):
    rgb = img_array.copy()
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    _, thresh = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
    coords = cv2.findNonZero(thresh)
    if coords is not None:
        x,y,w,h = cv2.boundingRect(coords); rgb = rgb[y:y+h, x:x+w]
    h2,w2 = rgb.shape[:2]; scale = size/max(h2,w2)
    nh,nw = int(round(h2*scale)), int(round(w2*scale))
    rgb = cv2.resize(rgb,(nw,nh),interpolation=cv2.INTER_AREA)
    pt=(size-nh)//2; pb=size-nh-pt; pl=(size-nw)//2; pr=size-nw-pl
    rgb = cv2.copyMakeBorder(rgb,pt,pb,pl,pr,cv2.BORDER_REFLECT_101)
    mask = np.zeros(rgb.shape[:2],np.uint8)
    cv2.circle(mask,(size//2,size//2),int(size*0.97/2),255,-1)
    rgb[mask==0]=0
    lab = cv2.cvtColor(rgb,cv2.COLOR_RGB2LAB)
    clahe = cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
    lab[:,:,0] = clahe.apply(lab[:,:,0])
    rgb = cv2.cvtColor(lab,cv2.COLOR_LAB2RGB)
    sig = max((size//10)|1,1)
    blur = cv2.GaussianBlur(rgb,(0,0),sigmaX=sig)
    rgb = cv2.addWeighted(rgb,4,blur,-4,128); rgb[mask==0]=0
    out = rgb.copy().astype(np.float32); out[:,:,1]=np.clip(out[:,:,1]*1.1,0,255)
    return out.astype(np.uint8)

def get_val_transform(img_size=IMG_SIZE):
    return A.Compose([A.Resize(img_size,img_size),
                      A.Normalize(mean=IMAGENET_MEAN,std=IMAGENET_STD),ToTensorV2()])

class GeM(nn.Module):
    def __init__(self,p=3,eps=1e-6):
        super().__init__(); self.p=nn.Parameter(torch.ones(1)*p); self.eps=eps
    def forward(self,x):
        return F.avg_pool2d(x.clamp(min=self.eps).pow(self.p),(x.size(-2),x.size(-1))).pow(1.0/self.p)

class DRModel(nn.Module):
    def __init__(self,backbone=BACKBONE,num_classes=5,dropout=0.5):
        super().__init__()
        self.backbone=timm.create_model(backbone,pretrained=False,features_only=False,num_classes=0,global_pool="")
        self.pool=GeM()
        self.head=nn.Sequential(nn.Flatten(),nn.BatchNorm1d(self.backbone.num_features),
            nn.Linear(self.backbone.num_features,256),nn.ReLU(inplace=True),
            nn.Dropout(dropout),nn.Linear(256,num_classes))
    def forward(self,x): return self.head(self.pool(self.backbone(x)))

def load_model(ckpt_path,num_classes=5,device=None):
    dev=device or DEVICE
    try: ckpt=torch.load(ckpt_path,map_location=dev,weights_only=False)
    except: ckpt=torch.load(ckpt_path,map_location=dev)
    model=DRModel(num_classes=num_classes); model.load_state_dict(ckpt["model_state"])
    return model.eval().to(dev)

@torch.no_grad()
def predict_image(img_rgb,model,device=None,use_tta=False):
    dev=device or DEVICE; img=preprocess_fundus(img_rgb)
    tf=get_val_transform()
    inp=tf(image=img)["image"].unsqueeze(0).to(dev)
    probs=F.softmax(model(inp),1).squeeze().cpu().numpy()
    return int(probs.argmax()),probs
""".strip()
(DEPLOY_DIR / "model_utils.py").write_text(model_utils_code)
print("✅ model_utils.py written.")

# ── app.py ────────────────────────────────────────────────────────
app_code = """
import os, numpy as np, matplotlib.pyplot as plt
from PIL import Image
import streamlit as st
from model_utils import load_model, predict_image, preprocess_fundus, GRADE_MAP, GRADE_COLORS, NUM_CLASSES, DEVICE

st.set_page_config(page_title="DR Grading v19", page_icon="🩺", layout="wide")
st.title("🩺 Diabetic Retinopathy Grading")
st.caption("EfficientNetV2-B1 + GeM + 5-Fold Ensemble | Production Pipeline v19")
st.warning("RESEARCH USE ONLY — NOT FOR CLINICAL DEPLOYMENT")

@st.cache_resource(show_spinner="Loading model...")
def get_model():
    base = os.path.dirname(os.path.abspath(__file__))
    ckpt = os.path.join(base, "best_model.pt")
    if not os.path.exists(ckpt): return None
    return load_model(ckpt, num_classes=5)

model = get_model()
if model is None:
    st.error("Model not found. Place best_model.pt here."); st.stop()

uploaded = st.file_uploader("Upload fundus image (JPG/PNG)", type=["jpg","jpeg","png"])
if uploaded:
    pil_img = Image.open(uploaded).convert("RGB")
    img_arr = np.array(pil_img)
    c1, c2 = st.columns(2)
    with c1: st.subheader("Original"); st.image(pil_img, use_column_width=True)
    with c2:
        st.subheader("Prediction")
        grade, probs = predict_image(img_arr, model)
        st.markdown(f"### Grade {grade}: {GRADE_MAP[grade]}")
        st.markdown(f"Confidence: {probs[grade]*100:.1f}%")
        fig, ax = plt.subplots(figsize=(5,3))
        ax.barh([f"G{i}" for i in range(NUM_CLASSES)], probs*100, color=GRADE_COLORS)
        ax.set_xlabel("Probability (%)"); ax.set_xlim(0,105)
        plt.tight_layout(); st.pyplot(fig); plt.close()
""".strip()
(DEPLOY_DIR / "app.py").write_text(app_code)
print("✅ app.py written.")

# ── requirements.txt ─────────────────────────────────────────────
req = "torch>=2.1\ntorchvision\ntimm>=1.0.0\nalbumentations>=1.4.0\nopencv-python-headless\nstreamlit>=1.35.0\ngrad-cam\npillow<11.0\nnumpy\n"
(DEPLOY_DIR / "requirements.txt").write_text(req)
print("✅ requirements.txt written.")

# ── README ────────────────────────────────────────────────────────
readme = "# DR Grading v19\n\nEfficientNetV2-B1 + GeM + 5-Fold Ensemble + TTA\n\n## Run\n```bash\npip install -r requirements.txt\nstreamlit run app.py\n```\n\n**RESEARCH USE ONLY**\n"
(DEPLOY_DIR / "README.md").write_text(readme)
print("✅ README.md written.")
print(f"\n📂 Deploy files → {DEPLOY_DIR}")
for f in sorted(DEPLOY_DIR.iterdir()):
    if f.is_file(): print(f"  {f.name}")

# ── Final Summary ─────────────────────────────────────────────────
state = st_load()
print("\n" + "=" * 68)
print("  DIABETIC RETINOPATHY GRADING — v19 FINAL SUMMARY")
print("=" * 68)
print(f"  Backbone : {BACKBONE}")
print(f"  Device   : {DEVICE.upper()}")
print(f"  Dataset  : APTOS 2019 ({len(df):,} images)")
print(f"  Strategy : 5-Fold CV + Ensemble × {len(tta_transforms_lst)} TTA views")
print()
if fold_val_qwks:
    for i, q in enumerate(fold_val_qwks):
        print(f"    Fold {i}: QWK={q:.4f}")
    print(f"    Mean: {np.mean(fold_val_qwks):.4f} ± {np.std(fold_val_qwks):.4f}")
print(f"    OOF QWK:       {state.get('oof_qwk', 'N/A')}")
print(f"    Optimised QWK: {state.get('opt_qwk', 'N/A')}")
tq = state.get("test_qwk"); ta = state.get("test_acc")
print(f"    Test QWK:      {tq if tq else 'N/A'}")
print(f"    Test Acc:      {float(ta)*100:.2f}%" if ta else "    Test Acc:      N/A")
print()
print(f"  Artifacts → {ARTIFACT_DIR}")
print(f"  Deploy    → {DEPLOY_DIR}")
print("=" * 68)
print("  ⚠️ RESEARCH USE ONLY — NOT FOR CLINICAL DEPLOYMENT")
print("=" * 68)